# Musanga V2 — Notebook 03 | Native UAV illumination harmonization

**Revision:** Compare native RGB (E0), **true site-fixed global gain** (E1), site gain plus **8 m local correction** (E2), site gain plus **25 m local correction** (E3), and legacy **per-patch global gain** (E4; diagnostic control).

All original native-resolution orthomosaics and labels stay unchanged. Sample patches and a coarse context overview are read into RAM only. E2/E3 estimate illumination using 8/25 m physical smoothing with extended surrounding context, rather than truncating broad filters at 512 px patch borders. Outputs exclusively: `MyDrive/Colab Notebooks/Musanga/KVO_work/V2/03_illumination_harmonization/`.

**Scientific limits:** RGB brightness is not calibrated irradiance; site-level gains are based on candidate windows rather than a full statistically representative mosaic survey. Luki remains external to training/reference fitting. True LOSO must derive reference statistics on the three training forests for each fold, treating unlabeled held-out-site adaptation as a separate method.

Sections **00–06** should be rerun in order after this revision.

## 00 — Setup and immutable UAV data sources

CPU-only. Read original native-resolution RGB mosaics; establish the V2-only destination and fixed experimental settings.

In [ ]:
%pip -q install numpy pandas rasterio scipy matplotlib geopandas pyogrio
from pathlib import Path
import numpy as np,pandas as pd,rasterio,matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, map_coordinates
from rasterio.windows import Window
from rasterio.enums import Resampling
from IPython.display import display
from google.colab import drive
import json,hashlib,math,warnings
MY=Path("/content/drive/MyDrive")
if not MY.exists():drive.mount("/content/drive")
ROOT=MY/"Colab Notebooks"/"Musanga"
GEO=ROOT/"data"/"Geo_Ref"
V2=ROOT/"KVO_work"/"V2"
OUT=V2/"03_illumination_harmonization"
OUT.mkdir(parents=True,exist_ok=True)
SITES={
 "MOS_1_2":"MOS_1_2_kvo.tif",
 "Yoko":"Yoko_kvo_georef.tif",
 "Yangambi":"Yangambi_georef.tif",
 "Baego":"Baego_georef.tif",
 "Luki":"Luki.tif"}
TRAIN_SITES=[s for s in SITES if s!="Luki"]
SEED=20261013
CANDIDATES_PER_SITE=100
PATCH=512
N_PER_STRATUM=4 # 4 dark, 4 middle and 4 bright per forest
MIN_VALID=.80
# Conservative, predeclared methods: no evaluation labels or performance used for tuning.
GLOBAL_GAIN_BOUNDS=(.85,1.15)
LOCAL_GAIN_BOUNDS=(.75,1.25)
LOCAL_SCALES_M=(8.0,25.0)
CONTEXT_RES_M=2.0 # coarse overview of surrounding mosaic, NOT resampling of source data on Drive
print("Run mode: READ ONLY inputs, CPU only. Output:",OUT)
for site,name in SITES.items():
    assert (GEO/name).exists(),f"Missing unchanged native UAV image: {GEO/name}"
assert OUT.is_relative_to(V2),"All V2 output MUST remain beneath KVO_work/V2"


## 01 — Stratified UAV RGB patch sampling

Select spatially distributed 512×512 crops. When a forest has a sparse valid footprint (notably Baego), use small RGB-only probes to locate valid regions and then verify full-size candidate patches. Preserve a common 0–1 brightness scale across sites and select dark/intermediate/bright images for the harmonization gallery.


In [ ]:
def valid_rgb03(rgb):
    return ~((rgb==0).all(axis=2)|(rgb==255).all(axis=2)|
             ((rgb[:,:,0]==0)&(rgb[:,:,1]==0)&(rgb[:,:,2]==255)))
def luminosity03(x):
    return (.2126*x[:,:,0]+.7152*x[:,:,1]+.0722*x[:,:,2])
def candidates03(height,width,n,patch,seed):
    rng=np.random.default_rng(seed)
    gy=max(2,int(np.ceil(np.sqrt(n*height/max(width,1)))))
    gx=max(2,int(np.ceil(n/gy)))
    for iy in range(gy):
        for ix in range(gx):
            row0=int(iy*max(0,height-patch)/gy)
            row1=int((iy+1)*max(0,height-patch)/gy)+1
            col0=int(ix*max(0,width-patch)/gx)
            col1=int((ix+1)*max(0,width-patch)/gx)+1
            r=int(rng.integers(row0,max(row0+1,row1)))
            c=int(rng.integers(col0,max(col0+1,col1)))
            yield max(0,min(height-patch,r)),max(0,min(width-patch,c))
RECORDS03=[];PATCHES03={};PIXEL_SIZE03={};CANDIDATE_ROWS03=[]
for si,(site,name) in enumerate(SITES.items()):
    available=[]
    with rasterio.open(GEO/name) as ds:
        assert ds.count>=3 and all(t=="uint8" for t in ds.dtypes[:3]),(site,ds.dtypes)
        assert ds.crs is not None and ds.crs.is_projected
        assert ds.width>=PATCH and ds.height>=PATCH
        gsd_x,gsd_y=abs(ds.transform.a),abs(ds.transform.e)
        gsd=float(np.sqrt(gsd_x*gsd_y))
        PIXEL_SIZE03[site]=gsd
        for k,(row,col) in enumerate(candidates03(ds.height,ds.width,CANDIDATES_PER_SITE,PATCH,SEED+si)):
            im=np.moveaxis(ds.read([1,2,3],window=Window(col,row,PATCH,PATCH)),0,-1)
            mask=valid_rgb03(im)
            if mask.mean()<MIN_VALID:continue
            x=im.astype(np.float32)/255.0
            median=float(np.median(luminosity03(x)[mask]))
            available.append(dict(row=row,col=col,median=median,
                                  valid_fraction=float(mask.mean())))
        # Fallback for sparse/discontinuous valid footprints (especially Baego):
        # Cheap 128 px RGB probes across random locations find candidate canopy regions,
        # then verify the *full* 512 px patch against the SAME >=80% validity gate.
        # Avoids wasting hundreds of expensive 512px reads on background/nodata.
        TARGET_VALID03=max(3,3*N_PER_STRATUM)
        if len(available)<TARGET_VALID03:
            rng=np.random.default_rng(SEED+1000+si)
            seen={(z["row"],z["col"]) for z in available}
            accepted_before=len(available)
            attempts=0
            while len(available)<TARGET_VALID03 and attempts<1200:
                attempts+=1
                row=int(rng.integers(0,ds.height-PATCH+1))
                col=int(rng.integers(0,ds.width-PATCH+1))
                if (row,col) in seen:continue
                seen.add((row,col))
                # Probe near center of proposed full window; use original RGB validity,
                # never unreliable Luki dataset_mask / fourth alpha band.
                probe_size=128
                probe=np.moveaxis(ds.read([1,2,3],window=Window(
                    col+(PATCH-probe_size)//2,row+(PATCH-probe_size)//2,
                    probe_size,probe_size)),0,-1)
                if valid_rgb03(probe).mean()<.94:continue
                im=np.moveaxis(ds.read([1,2,3],window=Window(col,row,PATCH,PATCH)),0,-1)
                mask=valid_rgb03(im)
                if mask.mean()<MIN_VALID:continue
                x=im.astype(np.float32)/255.
                available.append(dict(row=row,col=col,
                    median=float(np.median(luminosity03(x)[mask])),
                    valid_fraction=float(mask.mean())))
            print(site,"validity-aware fallback:",attempts,"small probes;",
                  len(available)-accepted_before,"additional full patches",flush=True)
        for z in available:
            CANDIDATE_ROWS03.append(dict(site=site,row=z["row"],col=z["col"],
                median_luminance=z["median"],valid_fraction=z["valid_fraction"]))
        if len(available)<3:
            raise RuntimeError(f"{site}: only {len(available)} valid candidates; cannot compare harmonization.")
        order=np.argsort([z["median"] for z in available])
        # disjoint brightness ranks for targeted but lightweight patch review
        strata={
            "dark":order[:min(N_PER_STRATUM,len(order)//3)],
            "middle":order[max(0,len(order)//2-N_PER_STRATUM//2):max(0,len(order)//2-N_PER_STRATUM//2)+N_PER_STRATUM],
            "bright":order[max(0,len(order)-N_PER_STRATUM):]}
        seen=set()
        for stratum,ii in strata.items():
            for p in ii:
                z=available[int(p)]
                key=(z["row"],z["col"])
                if key in seen:continue
                seen.add(key)
                im=np.moveaxis(ds.read([1,2,3],window=Window(z["col"],z["row"],PATCH,PATCH)),0,-1)
                valid=valid_rgb03(im)
                x=im.astype(np.float32)/255.
                pid=f"{site}_{z['row']}_{z['col']}"
                PATCHES03[pid]=(x,valid,site)
                RECORDS03.append(dict(patch_id=pid,site=site,row=z["row"],col=z["col"],
                    illumination_stratum=stratum,native_gsd_m=gsd,
                    median_luminance=z["median"],valid_fraction=z["valid_fraction"]))
    print(site,":",len(available),"valid candidates;",len([r for r in RECORDS03 if r["site"]==site]),
          "stratified patches; GSD",round(gsd,4),"m",flush=True)
CANDIDATE_TABLE03=pd.DataFrame(CANDIDATE_ROWS03)
CANDIDATE_TABLE03.to_csv(OUT/"03_valid_candidate_windows.csv",index=False)
PATCH_TABLE03=pd.DataFrame(RECORDS03)
PATCH_TABLE03.to_csv(OUT/"03_sampled_patch_registry.csv",index=False)
display(PATCH_TABLE03.groupby(["site","illumination_stratum"]).agg(
    patches=("patch_id","count"),median_brightness=("median_luminance","median"),
    gsd_m=("native_gsd_m","first")).round(3))


## 02 — Site-level luminance reference from candidate windows

Calculate **one fixed RGB luminance median per site**, using all available valid candidates rather than the intentionally selected dark/middle/bright gallery patches. The reference is the equal-site median from the **four development forests**; Luki is excluded from reference fitting. Sparse Baego sampling is flagged. These are image brightness indices, not radiometrically calibrated illumination; future strict LOSO references must be refitted using the three training forests only.

In [ ]:
# Candidate sampling is not a full-mosaic probability sample, but is less biased than
# deliberately selecting 4 dark + 4 middle + 4 bright windows for normalization.
# A single site gain will be applied to ALL selected patches from that site.
SITE_STATS03=CANDIDATE_TABLE03.groupby("site",as_index=False).agg(
    n_valid_candidate_windows=("median_luminance","count"),
    site_luminance_median=("median_luminance","median"),
    site_luminance_p10=("median_luminance",lambda z:float(z.quantile(.10))),
    site_luminance_p90=("median_luminance",lambda z:float(z.quantile(.90))))
SITE_STATS03["site_sampling_status"]=np.where(
    SITE_STATS03.n_valid_candidate_windows<20,"LOW_SAMPLE","SAMPLED")
assert set(SITE_STATS03.site)==set(SITES)
TRAIN_STATS03=SITE_STATS03.set_index("site").loc[TRAIN_SITES]
REFERENCE_LUMINANCE03=float(TRAIN_STATS03.site_luminance_median.median())
SITE_GAIN03={}
for _,row in SITE_STATS03.iterrows():
    SITE_GAIN03[row.site]=float(np.clip(
        REFERENCE_LUMINANCE03/max(float(row.site_luminance_median),1e-6),
        *GLOBAL_GAIN_BOUNDS))
SITE_STATS03["fixed_site_gain"]=[SITE_GAIN03[s] for s in SITE_STATS03.site]
SITE_STATS03.to_csv(OUT/"03_site_luminance_and_fixed_gains.csv",index=False)
display(SITE_STATS03.round(4))
print("Four-training-site reference index:",round(REFERENCE_LUMINANCE03,4))
print("Fixed site gains:",{s:round(v,4) for s,v in SITE_GAIN03.items()})
print("CAUTION: Site statistics from candidate windows, not a fully area-weighted mosaic survey.")
print("LOSO must refit reference using only 3 training sites; this stage is diagnostic.")


## 03 — Fixed site normalization, local illumination fields at 8/25 m and patch control

Five methods: **E0** unmodified RGB; **E1** one fixed site-wide luminance gain; **E2** E1 plus locally varying gain based on an 8 m illumination field; **E3** E1 plus a 25 m field; **E4** independent normalization of each 512 px patch (earlier method, retained only as a diagnostic control).

For E2 and E3 the illumination field is estimated from a **coarse, read-only overview of a larger geographic context**, including up to 75 m around each patch, so that 25 m Gaussian smoothing is not incorrectly truncated at the patch boundary. These overviews stay in memory and do not change source resolution or georeferencing. Local corrections are bounded, and their global anchor is the median of the sampled context.

In [ ]:
METHODS03=["E0_native","E1_site","E2_site_local8m","E3_site_local25m","E4_patch"]
def multiply03(x,mask,gain):
    corrected=np.clip(x*gain,0,1).astype(np.float32)
    corrected[~mask]=x[~mask]
    return corrected

def context_fields03(src,row,col,gsd):
    """Coarse illumination estimates with 3*25 m source halo.
    RGB overview is read-only and resampled IN MEMORY; source pixels are untouched.
    Returns a spatial field for each requested physical smoothing scale.
    """
    halo_px=int(np.ceil(3*max(LOCAL_SCALES_M)/gsd))
    native_width=PATCH+2*halo_px
    # Approximate 2 m overview sampling (consistent physical size across sites).
    out_side=max(64,int(np.ceil(native_width*gsd/CONTEXT_RES_M)))
    window=Window(col-halo_px,row-halo_px,native_width,native_width)
    im=src.read([1,2,3],window=window,
        out_shape=(3,out_side,out_side),
        boundless=True,fill_value=0,resampling=Resampling.nearest)
    coarse=np.moveaxis(im,0,-1)
    valid=valid_rgb03(coarse)
    light=luminosity03(coarse.astype(np.float32)/255.)
    # Pixel-to-pixel projected scale of overview; finite precision expected.
    coarse_res=native_width*gsd/out_side
    weights=valid.astype(np.float32)
    yy,xx=np.meshgrid(
        (halo_px+np.arange(PATCH,dtype=np.float32)+.5)*out_side/native_width-.5,
        (halo_px+np.arange(PATCH,dtype=np.float32)+.5)*out_side/native_width-.5,
        indexing="ij")
    coords=np.stack([yy,xx])
    fields={}
    for meters in LOCAL_SCALES_M:
        sigma=meters/coarse_res
        denominator=gaussian_filter(weights,sigma=sigma,mode="constant",cval=0)
        numerator=gaussian_filter(light*weights,sigma=sigma,mode="constant",cval=0)
        field=numerator/np.maximum(denominator,1e-3)
        support=map_coordinates(denominator,coords,order=1,mode="nearest")
        interp=map_coordinates(field,coords,order=1,mode="nearest").astype(np.float32)
        fields[meters]=(interp,support.astype(np.float32))
    return fields,dict(overview_resolution_m=float(coarse_res),
                       overview_width_px=int(out_side),context_halo_m=float(halo_px*gsd))

def local_correct03(x,mask,field,support):
    # Anchor to its OWN context, preserving E1's fixed site-wide gain.
    # Applying correction to E1*x commutes with raw illumination ratio below.
    supported=mask&(support>=.20)
    if supported.sum()<100:
        return x.copy(),dict(local_insufficient_support=True)
    anchor=float(np.median(field[supported]))
    ratio=anchor/np.maximum(field,.025)
    bounded=np.clip(ratio,*LOCAL_GAIN_BOUNDS).astype(np.float32)
    # No correction where coarse context support is insufficient.
    bounded[support<.20]=1.
    out=multiply03(x,mask,bounded[:,:,None])
    return out,dict(local_insufficient_support=False,
        local_gain_p05=float(np.quantile(bounded[mask],.05)),
        local_gain_p50=float(np.median(bounded[mask])),
        local_gain_p95=float(np.quantile(bounded[mask],.95)),
        local_gain_clipped_pct=float(100*np.mean(
          (ratio[supported]<LOCAL_GAIN_BOUNDS[0])|
          (ratio[supported]>LOCAL_GAIN_BOUNDS[1]))))

def transform03(x,mask,site,mode,fields=None):
    if mode=="E0_native":
        return x.copy(),dict(fixed_site_gain=1.,local_scale_m=0.)
    if mode=="E4_patch":
        lum=luminosity03(x)[mask]
        patch_gain=float(np.clip(REFERENCE_LUMINANCE03/max(float(np.median(lum)),1e-6),
                                 *GLOBAL_GAIN_BOUNDS))
        return multiply03(x,mask,patch_gain),dict(patch_gain=patch_gain,local_scale_m=0.)
    fixed_gain=SITE_GAIN03[site]
    global_img=multiply03(x,mask,fixed_gain)
    if mode=="E1_site":
        return global_img,dict(fixed_site_gain=fixed_gain,local_scale_m=0.)
    scale={"E2_site_local8m":8.0,"E3_site_local25m":25.0}.get(mode)
    if scale is None:raise ValueError(mode)
    assert fields is not None and scale in fields, "Read extended native-mosaic context first."
    result,info=local_correct03(global_img,mask,*fields[scale])
    return result,dict(fixed_site_gain=fixed_gain,local_scale_m=scale,**info)


## 04 — Objective QA: distribution, texture, clipping and colour stability

Quantify before/after brightness, local luminance variability, chromaticity, edge preservation, saturation, clipping and gain limits. These measurements reveal *how much* imagery changes but do not prove better species identification. Aggressive corrections will be rejected before model training.

In [ ]:
def qa03(original,corrected,valid):
    a=luminosity03(original)[valid];b=luminosity03(corrected)[valid]
    clipped=np.any((corrected[valid]>=.999)&(original[valid]<.999),axis=1)
    # Normalized colour coordinates; safe for low-brightness values.
    chrom_a=original[valid]/np.maximum(original[valid].sum(axis=1,keepdims=True),1e-5)
    chrom_b=corrected[valid]/np.maximum(corrected[valid].sum(axis=1,keepdims=True),1e-5)
    chrom_shift=np.mean(np.abs(chrom_a-chrom_b))
    # Neighbor differences, restricted to interior valid pixels.
    lum_a=luminosity03(original);lum_b=luminosity03(corrected)
    gx_a=lum_a[:,1:]-lum_a[:,:-1]
    gx_b=lum_b[:,1:]-lum_b[:,:-1]
    vx=valid[:,1:]&valid[:,:-1]
    texture_ratio=float(np.std(gx_b[vx])/max(np.std(gx_a[vx]),1e-6))
    texture_corr=float(np.corrcoef(gx_a[vx],gx_b[vx])[0,1]) if vx.sum()>10 and np.std(gx_a[vx])>1e-6 and np.std(gx_b[vx])>1e-6 else np.nan
    return dict(median_after=float(np.median(b)),median_before=float(np.median(a)),
        luminance_p10=float(np.quantile(b,.10)),luminance_p90=float(np.quantile(b,.90)),
        luminance_sd=float(b.std()),brightness_change=float(np.median(b)-np.median(a)),
        introduced_saturated_pixel_pct=float(100*clipped.mean()),
        mean_abs_chromaticity_shift=float(chrom_shift),
        local_edge_std_ratio=texture_ratio,local_edge_correlation=texture_corr)
RESULT03=[];PREVIEWS03={}
for site,name in SITES.items():
    subset=PATCH_TABLE03[PATCH_TABLE03.site==site]
    # Reuse open raster once per site; overview halos are computed once per patch
    # for BOTH 8 m and 25 m fields.
    with rasterio.open(GEO/name) as src:
        for _,row in subset.iterrows():
            pid=row.patch_id
            original,mask,_=PATCHES03[pid]
            context,context_info=context_fields03(src,int(row.row),int(row.col),PIXEL_SIZE03[site])
            PREVIEWS03[pid]={}
            for method in METHODS03:
                transformed,params=transform03(original,mask,site,method,context)
                stats=qa03(original,transformed,mask)
                prefixed_params={f"operator_{k}":v for k,v in params.items()}
                RESULT03.append(dict(patch_id=pid,site=site,stratum=row.illumination_stratum,
                    method=method,valid_fraction=float(mask.mean()),
                    **stats,**prefixed_params,**context_info))
                PREVIEWS03[pid][method]=transformed
    print("Corrected and audited",site,":",len(subset),"patches",flush=True)
QA_TABLE03=pd.DataFrame(RESULT03)
QA_TABLE03.to_csv(OUT/"03_harmonization_patch_metrics.csv",index=False)
AGG03=QA_TABLE03.groupby(["site","method"],as_index=False).agg(
    n=("patch_id","count"),median_luminance=("median_after","median"),
    mean_clipped_pct=("introduced_saturated_pixel_pct","mean"),
    median_chromaticity_shift=("mean_abs_chromaticity_shift","median"),
    median_edge_ratio=("local_edge_std_ratio","median"),
    median_edge_corr=("local_edge_correlation","median"))
AGG03.to_csv(OUT/"03_harmonization_site_summary.csv",index=False)
display(AGG03.round(4))
print("NOTE: Numeric QA here tests image transformation only; crown accuracy is assessed later.")


## 05 — Before/after diagnostic galleries

Show dark, medium and bright sampled patches for each forest on **identical RGB display scales** and save one gallery per site. Avoid automatic display normalization, which would conceal brightness differences.

In [ ]:
for site in SITES:
    chosen=[]
    for st in ("dark","middle","bright"):
        sub=PATCH_TABLE03[(PATCH_TABLE03.site==site)&(PATCH_TABLE03.illumination_stratum==st)]
        if len(sub):chosen.append(sub.iloc[len(sub)//2].patch_id)
    fig,axs=plt.subplots(len(chosen),len(METHODS03),figsize=(18,3.4*len(chosen)),squeeze=False,
                         constrained_layout=True)
    for i,pid in enumerate(chosen):
        for j,method in enumerate(METHODS03):
            axs[i,j].imshow(PREVIEWS03[pid][method],vmin=0,vmax=1)
            axs[i,j].set_title(f"{site} | {PATCH_TABLE03.set_index('patch_id').loc[pid,'illumination_stratum']} | {method}")
            axs[i,j].axis("off")
    figure=OUT/f"03_{site}_before_after_gallery.png"
    fig.savefig(figure,dpi=140,bbox_inches="tight")
    display(fig);plt.close(fig)
    print("Saved gallery:",figure,flush=True)


## 06 — Reproducibility contract and deployment safeguards

Record site-fixed gains, all operator bounds and 8/25 m physical smoothing scales, plus sample and context settings. Outputs remain under `Musanga/KVO_work/V2/03_illumination_harmonization/`. **This is preprocessing image QA, not model validation.** For a strict four-forest LOSO experiment, recalibrate cross-site reference gains using only the three training sites; report any unsupervised test-image adaptation separately.

In [ ]:
# Diagnostic site fixed gains are computed for this image QA run.
# In strict four-site LOSO: use 3 training forests to set the REFERENCE;
# test site's image-only statistics for its own gain = unsupervised target adaptation.
CONTRACT03=dict(
 notebook="V2/03 UAV Illumination Harmonization",
 output_root=str(OUT),
 authoritative_uav={s:str(GEO[name]) for s,name in SITES.items()},
 methods=METHODS03,
 seed=SEED,patch_size_px=PATCH,patches_per_brightness_stratum=N_PER_STRATUM,
 global_reference_training_sites=TRAIN_SITES,
 reference_luminance=REFERENCE_LUMINANCE03,
 candidate_sample_counts={str(r.site):int(r.n_valid_candidate_windows) for _,r in SITE_STATS03.iterrows()},
 fixed_site_gains=SITE_GAIN03,
 gain_bounds=list(GLOBAL_GAIN_BOUNDS),local_gain_bounds=list(LOCAL_GAIN_BOUNDS),
 local_scales_metres=list(LOCAL_SCALES_M),
 local_context_halo_metres=3*max(LOCAL_SCALES_M),
 local_overview_resolution_metres=CONTEXT_RES_M,
 local_field="valid-weighted 2 m overview, Gaussian smooth with ~75 m context",
 patch_gain_control="E4 only; diagnostic, not equivalent to site-level harmonization",
 images="unchanged native UAV mosaic pixels and geometry",
 selection_policy="No selection based on Luki independent evaluation labels",
 four_site_LOSO="Reference recomputed per training triple; target-image unsupervised gain is a separately reported adaptation scenario",
 caution="Brightness index is not calibrated incident light; candidate windows may not represent whole mosaic",
 output_products=["03_valid_candidate_windows.csv","03_site_luminance_and_fixed_gains.csv",
   "03_harmonization_patch_metrics.csv","03_harmonization_site_summary.csv",
   "03_<site>_before_after_gallery.png"])
(OUT/"03_harmonization_contract.json").write_text(json.dumps(CONTRACT03,indent=2))
print("V2 NOTEBOOK 03 COMPLETE:",OUT)
print("Compare image changes visually and numerically BEFORE any U-Net training.")
print("Do not generate full-size corrected orthomosaics yet.")
